# Exp 4 Reproducibility Check

This notebook reruns `code/04_experiment4_cv_factorial.R` in an isolated scratch
workspace, compares the rerun fold structure and numeric outputs against
`expected/exp4_cv_factorial.rds`, and compares the regenerated AUC plot against
the same plot rebuilt from the expected baseline result.

In [ ]:
suppressPackageStartupMessages({
  library(survey)
  library(tidyverse)
  library(xgboost)
  library(pROC)
})

helper_candidates <- c("repro_utils.R", file.path("notebooks", "repro", "repro_utils.R"))
helper_path <- helper_candidates[file.exists(helper_candidates)][1]
if (is.na(helper_path)) {
  stop("repro_utils.R not found.")
}
source(helper_path)

baseline_result_path <- path_in_repo("expected", "exp4_cv_factorial.rds")
scratch_dir <- new_scratch("exp4_repro")

cat("Scratch workspace:", scratch_dir, "\n")
cat("Baseline result:", baseline_result_path, "\n")

In [ ]:
run_script_in_scratch(
  "code/04_experiment4_cv_factorial.R",
  scratch_dir,
  c("data/model_df.rds")
)

baseline_exp4 <- readRDS(baseline_result_path)
rerun_exp4 <- readRDS(file.path(scratch_dir, "results", "exp4_cv_factorial.rds"))

In [ ]:
params_to_df <- function(exp4_obj) {
  tibble(
    name = c(
      "n_folds",
      "n_repeats",
      "nrounds",
      paste0("xgb_params.", names(exp4_obj$params$xgb_params))
    ),
    value = c(
      as.character(exp4_obj$params$n_folds),
      as.character(exp4_obj$params$n_repeats),
      as.character(exp4_obj$params$nrounds),
      vapply(exp4_obj$params$xgb_params, as.character, character(1))
    )
  )
}

fold_profile <- function(exp4_obj) {
  exp4_obj$results %>%
    distinct(cv_type, repeat_id, fold_id, n_test, n_pos) %>%
    arrange(cv_type, repeat_id, fold_id)
}

ordered_results <- function(exp4_obj) {
  exp4_obj$results %>%
    arrange(cv_type, repeat_id, fold_id, train_type, eval_type)
}

ordered_summary <- function(exp4_obj) {
  exp4_obj$summary_stats %>%
    arrange(cv_type, train_type, eval_type)
}

exp4_checks <- list(
  compare_df_exact("exp4_params", params_to_df(rerun_exp4), params_to_df(baseline_exp4)),
  compare_df_exact("exp4_fold_profile", fold_profile(rerun_exp4), fold_profile(baseline_exp4)),
  compare_df_tol("exp4_results", ordered_results(rerun_exp4), ordered_results(baseline_exp4), tol = 1e-12),
  compare_df_tol("exp4_summary_stats", ordered_summary(rerun_exp4), ordered_summary(baseline_exp4), tol = 1e-12)
)

bind_rows(exp4_checks)

In [ ]:
build_exp4_auc_plot <- function(exp4_obj) {
  plot_df <- exp4_obj$results %>%
    mutate(condition = paste0(train_type, " Train\n", eval_type, " Eval"))

  ggplot(plot_df, aes(x = condition, y = auc, fill = cv_type)) +
    geom_violin(position = position_dodge(0.8), alpha = 0.6, trim = FALSE) +
    geom_boxplot(position = position_dodge(0.8), width = 0.15, outlier.size = 0.8) +
    scale_fill_manual(values = c("Random" = "#4393C3", "PSU" = "#D6604D"),
                      name = "CV Type") +
    labs(x = NULL, y = "AUC",
         title = "Cross-Validation Type Effect on AUC Estimates") +
    theme_bw(base_size = 12) +
    theme(axis.text.x = element_text(size = 9),
          legend.position = "top")
}

rerun_plot_path <- file.path(scratch_dir, "out", "fig_exp4_cv_type_auc.pdf")
baseline_plot_path <- file.path(scratch_dir, "baseline_out", "fig_exp4_cv_type_auc.pdf")

ggsave(rerun_plot_path, build_exp4_auc_plot(rerun_exp4), width = 8, height = 5)
ggsave(baseline_plot_path, build_exp4_auc_plot(baseline_exp4), width = 8, height = 5)

figure_checks <- list(
  compare_pdf_figure("fig_exp4_cv_type_auc", rerun_plot_path, baseline_plot_path)
)

bind_rows(figure_checks)

In [ ]:
exp4_summary <- summarize_results(c(exp4_checks, figure_checks))
exp4_summary$results